# 🧠 Resume Intelligence and Job-Matching Copilot
### A Beginner-Friendly End-to-End NLP + Generative AI Project (Google Colab)

**What this notebook does:**
This notebook builds an AI recruitment assistant step by step. It will:
- Read a resume and a job description
- Clean and process the text using NLP
- Extract skills, education, experience, etc.
- Compare resume skills vs job skills
- Calculate a semantic "match score" using AI embeddings
- Rank the candidate
- Use Generative AI to write a summary, strengths/weaknesses, and interview questions
- Produce a final "Explainable AI" report

**How to use this notebook:**
1. Open this file in **Google Colab** (File → Upload notebook, or drag-and-drop).
2. Run the cells **one at a time, from top to bottom**, using the ▶️ button.
3. When a cell asks you to upload a file, upload your **resume** (PDF or TXT) and the **job description** (PDF or TXT).
4. Read the explanation below each cell — it tells you *what* the code does, *why* we need it, and *what output to expect*.

> 💡 Every code cell is small on purpose. Small cells are easier to understand and easier to debug.


---
## 🚦 PHASE 1 — Project Setup

### Key concepts (in plain English)

- **NLP (Natural Language Processing):** A branch of AI that helps computers read, understand, and work with human language (like English text in a resume).
- **Resume Parsing:** The process of taking a messy resume file and pulling out useful structured information from it (name, skills, education, etc.) — like turning a paragraph into a neat table.
- **Job Matching:** Comparing a candidate's resume to a job description to see how well they fit — similar to how a recruiter skims a resume and asks "does this person have what the job needs?"

We'll build this project in small, understandable pieces.


### Cell 1: Install libraries

In [ ]:
# We install (download) the extra Python libraries this project needs.
# Colab already has many libraries, but not all of them, so we install the missing ones.

!pip install pdfplumber nltk sentence-transformers scikit-learn -q
# pdfplumber          -> lets us read text out of PDF files
# nltk                -> classic NLP toolkit (tokenizing, stopwords, stemming, lemmatizing)
# sentence-transformers -> turns sentences into number vectors (embeddings) for AI comparison
# scikit-learn        -> gives us the cosine similarity function and other ML tools
# -q means "quiet" so it doesn't print a huge wall of text


**What this cell does:** Downloads the Python packages we need.
**Why we use it:** Without these libraries, we can't read PDFs, clean text, or use AI models.
**Expected output:** Some installation progress text, ending with no errors (warnings are OK).


### Cell 2: Import libraries

In [ ]:
# "Importing" means loading a library into memory so we can use its functions.

import re                     # re = "regular expressions", used for pattern matching in text (e.g. removing numbers)
import string                 # gives us a ready-made list of punctuation characters
import pandas as pd           # pandas lets us build and display neat tables
import nltk                   # our main NLP toolkit
import pdfplumber              # to read text from PDF files

# Download small NLTK data packages we will use later (tokenizer, stopwords, lemmatizer)
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')

print("✅ Libraries imported successfully")


**What this cell does:** Loads all the tools we installed so Python can use them.
**Why we use it:** You must import a library before using any of its functions, the same way you must open a toolbox before using a tool.
**Expected output:** Some short download confirmation lines, then `✅ Libraries imported successfully`.


### Cell 3: Upload your resume file

Click the "Choose Files" button that appears after running the cell, and upload **your resume** (PDF or TXT).


In [ ]:
from google.colab import files  # special Colab tool for uploading files from your computer

print("Please upload your RESUME file (PDF or TXT)")
uploaded_resume = files.upload()          # opens a file picker in Colab
resume_filename = list(uploaded_resume.keys())[0]   # grabs the name of the file you uploaded
print("Uploaded resume file:", resume_filename)


**What this cell does:** Opens a file-upload button and stores the resume file you choose.
**Why we use it:** The notebook needs your actual resume file to analyze it.
**Expected output:** An upload button, then a line like `Uploaded resume file: my_resume.pdf`.


### Cell 4: Upload the job description file

Now upload the **job description** you want to match your resume against. You can upload it as a `.txt` file, or a `.pdf`, or simply paste the text in the next cell instead.


In [ ]:
print("Please upload the JOB DESCRIPTION file (PDF or TXT)")
uploaded_jd = files.upload()
jd_filename = list(uploaded_jd.keys())[0]
print("Uploaded job description file:", jd_filename)


**What this cell does:** Lets you upload the job description file the same way you uploaded your resume.
**Why we use it:** We need something to compare the resume against — that's the job description.
**Expected output:** An upload button, then a line like `Uploaded job description file: jd.txt`.

> 📝 **Tip:** If you don't have a JD file handy, skip this cell and instead paste the job description text directly into the `jd_text` variable in Phase 2.


---
## 📄 PHASE 2 — Text Extraction

### Key concepts

- **Raw Text:** The plain, unprocessed text exactly as it appears in the document — no cleaning done yet.
- **Unstructured Data:** Data that doesn't follow a neat row/column format (like a resume paragraph), as opposed to **structured data** (like a spreadsheet).

Our first job is simply to pull the raw text out of the files.


### Cell 5: Function to read text from PDF or TXT

In [ ]:
def extract_text(filename):
    """Reads a file and returns its text content.
    Works for both .pdf and .txt files."""

    if filename.lower().endswith(".pdf"):
        text = ""
        with pdfplumber.open(filename) as pdf:   # open the PDF
            for page in pdf.pages:                # go through every page
                page_text = page.extract_text()    # pull text from that page
                if page_text:                       # only add it if something was found
                    text += page_text + "\n"
        return text
    else:  # it's a .txt file
        with open(filename, "r", encoding="utf-8", errors="ignore") as f:
            return f.read()

print("✅ extract_text() function is ready")


**What this cell does:** Defines a reusable function that can read either a PDF or a TXT file and return its text.
**Why we use it:** Writing this once as a function means we can reuse it for both the resume and the job description, instead of repeating code.
**Expected output:** `✅ extract_text() function is ready`


### Cell 6: Read the resume text

In [ ]:
resume_text = extract_text(resume_filename)   # call our function on the resume file
print("Resume text length:", len(resume_text), "characters")


**What this cell does:** Extracts the raw text from your uploaded resume and stores it in the variable `resume_text`.
**Why we use it:** We need the text in Python (not just in a file) so we can process it.
**Expected output:** Something like `Resume text length: 2350 characters`.


### Cell 7: Read the job description text

In [ ]:
jd_text = extract_text(jd_filename)   # call our function on the job description file
print("Job description text length:", len(jd_text), "characters")

# If you didn't upload a JD file, comment out the two lines above and instead paste text below:
# jd_text = """Paste the job description text here"""


**What this cell does:** Extracts the raw text from the job description file.
**Why we use it:** We need both texts side by side to compare them later.
**Expected output:** A character-count line similar to Cell 6.


### Cell 8: Display the raw text (preview)

In [ ]:
print("----- RESUME PREVIEW (first 500 characters) -----")
print(resume_text[:500])

print("\n----- JOB DESCRIPTION PREVIEW (first 500 characters) -----")
print(jd_text[:500])


**What this cell does:** Prints the first 500 characters of each document, just to sanity-check that the text was read correctly.
**Why we use it:** Always look at your raw data before processing it — this catches upload mistakes early.
**Expected output:** Two blocks of text showing the beginning of your resume and job description.


---
## 🧹 PHASE 3 — Text Cleaning

### Key concepts

- **Text Preprocessing:** Cleaning up messy raw text (removing punctuation, extra spaces, etc.) before feeding it to an NLP model.
- **Why cleaning is important:** Computers treat `"Python,"` and `"python"` as two *different* words unless we clean the text. Cleaning makes matching and comparison much more accurate.

We'll clean the text in small, separate steps so you can see exactly what each step changes.


### Cell 9: Lowercase conversion

In [ ]:
def to_lowercase(text):
    # .lower() converts every uppercase letter to lowercase
    # "Python" and "PYTHON" and "python" should all be treated as the same word
    return text.lower()

resume_clean = to_lowercase(resume_text)
jd_clean = to_lowercase(jd_text)

print(resume_clean[:200])   # preview the change


**What this cell does:** Converts all text to lowercase.
**Why we use it:** So the computer treats "Python" and "python" as the same word.
**Expected output:** A text preview that's now entirely in lowercase letters.


### Cell 10: Remove punctuation

In [ ]:
def remove_punctuation(text):
    # string.punctuation is a ready-made string of symbols like !"#$%&'()*+,-./:;<=>?@[\]^_`{|}~
    # str.maketrans creates a translation table that maps each punctuation character to None (deletes it)
    translator = str.maketrans('', '', string.punctuation)
    return text.translate(translator)

resume_clean = remove_punctuation(resume_clean)
jd_clean = remove_punctuation(jd_clean)

print(resume_clean[:200])


**What this cell does:** Strips out punctuation marks like commas, periods, and parentheses.
**Why we use it:** Punctuation usually adds noise, not meaning, for tasks like keyword matching.
**Expected output:** The same preview text, now without punctuation symbols.


### Cell 11: Remove special characters

In [ ]:
def remove_special_characters(text):
    # This regular expression keeps only letters, numbers, and spaces, and removes everything else
    # (things like bullet symbols •, emojis, or stray symbols that punctuation removal might have missed)
    return re.sub(r'[^a-zA-Z0-9\s]', '', text)

resume_clean = remove_special_characters(resume_clean)
jd_clean = remove_special_characters(jd_clean)

print(resume_clean[:200])


**What this cell does:** Removes any leftover odd symbols (like bullet points • or emojis) that aren't letters or numbers.
**Why we use it:** Resumes often have bullet characters and formatting symbols copied from Word/PDF that aren't real words.
**Expected output:** Cleaner text with no stray symbols.


### Cell 12: Remove numbers

In [ ]:
def remove_numbers(text):
    # \d means "any digit" (0-9). This regex replaces every digit with nothing.
    return re.sub(r'\d+', '', text)

# NOTE: We keep a numbers-included copy too, since experience (e.g. "3 years") needs numbers later!
resume_clean_no_numbers = remove_numbers(resume_clean)

print(resume_clean_no_numbers[:200])


**What this cell does:** Creates a version of the text with all numbers removed.
**Why we use it:** For tasks like word-frequency or skill-keyword matching, numbers usually aren't useful. But we keep the original (`resume_clean`) too, because later we'll need numbers to detect things like "5 years of experience".
**Expected output:** Text preview with digits removed.


### Cell 13: Remove extra spaces

In [ ]:
def remove_extra_spaces(text):
    # \s+ means "one or more whitespace characters" -> replace with a single space
    return re.sub(r'\s+', ' ', text).strip()

resume_clean = remove_extra_spaces(resume_clean)
jd_clean = remove_extra_spaces(jd_clean)

print(resume_clean[:200])
print("\n✅ Text cleaning complete")


**What this cell does:** Collapses multiple spaces/newlines into a single space and trims leading/trailing spaces.
**Why we use it:** PDFs often produce messy spacing (extra line breaks, tabs). This tidies it up.
**Expected output:** Neatly spaced text ending with `✅ Text cleaning complete`.


---
## 🔤 PHASE 4 — NLP Processing (using NLTK)

### Key concepts (with tiny examples)

- **Tokenization:** Splitting text into individual pieces (tokens), usually words.
  *Example:* `"I love Python"` → `["I", "love", "Python"]`
- **Stopword Removal:** Removing very common words that don't carry much meaning (like "the", "is", "and").
  *Example:* `"I am a data analyst"` → `["data", "analyst"]`
- **Stemming:** Chopping words down to a rough root form, sometimes crudely.
  *Example:* `"running"`, `"runs"`, `"ran"` → all might become `"run"` (or even `"runn"`).
- **Lemmatization:** Like stemming, but smarter — it reduces a word to its real dictionary base form.
  *Example:* `"better"` → `"good"`, `"running"` → `"run"`


### Cell 14: Tokenization

In [ ]:
from nltk.tokenize import word_tokenize

resume_tokens = word_tokenize(resume_clean)   # split resume text into individual words
jd_tokens = word_tokenize(jd_clean)           # split job description text into individual words

print("First 20 resume tokens:", resume_tokens[:20])


**What this cell does:** Breaks the cleaned text into a list of individual words (tokens).
**Why we use it:** Almost every NLP step after this works on individual words, not one big blob of text.
**Expected output:** A Python list of words, e.g. `['john', 'doe', 'data', 'analyst', ...]`


### Cell 15: Stopword removal

In [ ]:
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))   # a ready-made list of common English filler words

def remove_stopwords(tokens):
    # Keep only the words that are NOT in the stopword list
    return [word for word in tokens if word not in stop_words]

resume_tokens = remove_stopwords(resume_tokens)
jd_tokens = remove_stopwords(jd_tokens)

print("First 20 resume tokens after stopword removal:", resume_tokens[:20])


**What this cell does:** Removes common filler words like "the", "is", "and", "a" from our token lists.
**Why we use it:** These words appear in almost every document and don't help us tell resumes or skills apart.
**Expected output:** A shorter, more meaningful word list.


### Cell 16: Stemming

In [ ]:
from nltk.stem import PorterStemmer

stemmer = PorterStemmer()

# apply stemming to just the first 20 words, as a demonstration
sample_stems = [stemmer.stem(word) for word in resume_tokens[:20]]
print("Original :", resume_tokens[:20])
print("Stemmed  :", sample_stems)


**What this cell does:** Shows what stemming looks like on a small sample of words.
**Why we use it:** It's important to *see* stemming in action — notice some words get cut in a rough, non-dictionary way (e.g. "analytical" → "analyt"). We show this for learning purposes; we will actually use lemmatization (next cell) for our real pipeline since it's more accurate.
**Expected output:** Two lists side-by-side: original words and their stemmed versions.


### Cell 17: Lemmatization

In [ ]:
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()

def lemmatize_tokens(tokens):
    return [lemmatizer.lemmatize(word) for word in tokens]

resume_tokens = lemmatize_tokens(resume_tokens)
jd_tokens = lemmatize_tokens(jd_tokens)

print("First 20 resume tokens after lemmatization:", resume_tokens[:20])

# Rebuild clean strings from the final tokens - we'll use these strings later
resume_final_text = " ".join(resume_tokens)
jd_final_text = " ".join(jd_tokens)

print("\n✅ NLP preprocessing complete")


**What this cell does:** Reduces words to their proper dictionary root form, then rebuilds a clean text string from the processed tokens.
**Why we use it:** Lemmatization is more accurate than stemming for real applications, since it produces real words. `resume_final_text` and `jd_final_text` are what we'll use going forward.
**Expected output:** A lemmatized token list, then `✅ NLP preprocessing complete`.


---
## 🗂️ PHASE 5 — Information Extraction

### Key concepts

- **Named Entity Recognition (NER):** An NLP technique that finds and labels specific "entities" in text — like names, dates, organizations, or skills. Advanced NER uses AI models; we'll use a simpler beginner-friendly approach: **keyword matching against known skill lists**, which is easier to understand and still effective for resumes.
- **Information Extraction:** The general task of pulling structured facts (skills, education, experience) out of unstructured text.

We'll build a small "skills dictionary" and search for those words in the resume and job description. Feel free to expand this list with more skills relevant to your field!


### Cell 18: Define a skills dictionary

In [ ]:
# A simple, beginner-friendly list of common tech/analytics skills.
# You can add more keywords relevant to your own field!

SKILLS_DB = [
    "python", "sql", "excel", "power bi", "tableau", "machine learning",
    "deep learning", "nlp", "data analysis", "data visualization",
    "statistics", "pandas", "numpy", "scikit-learn", "tensorflow",
    "pytorch", "aws", "azure", "gcp", "spark", "hadoop", "java",
    "javascript", "html", "css", "django", "flask", "git", "docker",
    "kubernetes", "rest api", "etl", "data cleaning", "communication",
    "leadership", "teamwork", "problem solving"
]

print(f"✅ Skills dictionary ready with {len(SKILLS_DB)} skills")


**What this cell does:** Creates a list of skill keywords we'll search for.
**Why we use it:** This is the simplest possible way to detect skills without needing a complex AI model — we just check "does this word appear in the text?"
**Expected output:** `✅ Skills dictionary ready with 35 skills`


### Cell 19: Function to extract skills from text

In [ ]:
def extract_skills(text, skills_db=SKILLS_DB):
    text = text.lower()
    found_skills = []
    for skill in skills_db:
        if skill in text:              # simple substring search
            found_skills.append(skill)
    return sorted(set(found_skills))    # remove duplicates and sort alphabetically

resume_skills = extract_skills(resume_clean)
jd_skills = extract_skills(jd_clean)

print("Skills found in RESUME:", resume_skills)
print("Skills found in JOB DESCRIPTION:", jd_skills)


**What this cell does:** Searches both documents for any of the skills in our dictionary.
**Why we use it:** This gives us two clean lists — `resume_skills` and `jd_skills` — that we'll compare in Phase 6.
**Expected output:** Two printed Python lists of matched skills.


### Cell 20: Extract education, experience, and certifications (keyword-based)

In [ ]:
# Beginner-friendly keyword search for a few other common resume sections.
# This isn't perfect, but it's simple to understand and a great starting point.

EDUCATION_KEYWORDS = ["bachelor", "master", "b.tech", "m.tech", "bsc", "msc",
                      "mba", "phd", "degree", "university", "college"]
CERT_KEYWORDS = ["certified", "certification", "certificate"]

def extract_by_keywords(text, keyword_list):
    text = text.lower()
    return sorted(set([kw for kw in keyword_list if kw in text]))

resume_education = extract_by_keywords(resume_clean, EDUCATION_KEYWORDS)
resume_certifications = extract_by_keywords(resume_clean, CERT_KEYWORDS)

# Simple experience detector: look for patterns like "3 years" or "5+ years"
experience_matches = re.findall(r'(\d+)\+?\s*year', resume_text.lower())
resume_experience_years = max([int(x) for x in experience_matches], default=0)

print("Education keywords found:", resume_education)
print("Certification keywords found:", resume_certifications)
print("Detected years of experience:", resume_experience_years)


**What this cell does:** Looks for education-related words, certification-related words, and a "X years" pattern in the resume.
**Why we use it:** These are common resume sections recruiters care about, and simple keyword/pattern matching is an easy beginner-level way to detect them.
**Expected output:** Three lines showing whatever was found (lists may be empty if the resume doesn't mention them, and experience defaults to `0` if no pattern was found).


### Cell 21: Display extraction results in a table

In [ ]:
summary_table = pd.DataFrame({
    "Category": ["Skills", "Education Keywords", "Certifications", "Experience (years)"],
    "Extracted from Resume": [
        ", ".join(resume_skills) if resume_skills else "None found",
        ", ".join(resume_education) if resume_education else "None found",
        ", ".join(resume_certifications) if resume_certifications else "None found",
        resume_experience_years
    ]
})

summary_table   # in Colab, just typing a DataFrame name displays it as a nice table


**What this cell does:** Puts everything we extracted into one neat table using pandas.
**Why we use it:** Tables are much easier to read than scattered print statements.
**Expected output:** A styled table with 4 rows summarizing what was found in the resume.


---
## 🔍 PHASE 6 — Skill Matching

Now we compare the resume's skills against the job description's required skills.

- **Matched Skills:** Skills that appear in BOTH the resume and the job description — these are your strengths for this role.
- **Missing Skills:** Skills the job wants that are NOT in your resume — these are gaps to work on.
- **Extra Skills:** Skills in your resume that the job description didn't ask for — good bonus skills.


### Cell 22: Compare resume skills vs job skills

In [ ]:
resume_skills_set = set(resume_skills)
jd_skills_set = set(jd_skills)

matched_skills = sorted(resume_skills_set & jd_skills_set)   # skills in BOTH (set intersection)
missing_skills = sorted(jd_skills_set - resume_skills_set)   # in JD but NOT in resume
extra_skills = sorted(resume_skills_set - jd_skills_set)     # in resume but NOT in JD

print("✅ Matched Skills:", matched_skills)
print("❌ Missing Skills:", missing_skills)
print("➕ Extra Skills:", extra_skills)


**What this cell does:** Uses Python **set operations** to find overlapping and non-overlapping skills.
**Why we use it:** Set intersection (`&`) and difference (`-`) are the simplest, fastest way to compare two lists of items.
**Expected output:** Three printed lists: matched, missing, and extra skills.


### Cell 23: Skill match score and display

In [ ]:
if len(jd_skills_set) > 0:
    skill_match_score = round((len(matched_skills) / len(jd_skills_set)) * 100, 2)
else:
    skill_match_score = 0.0

print(f"📊 Skill Match Score: {skill_match_score}%")

skills_table = pd.DataFrame({
    "Type": ["Matched ✅", "Missing ❌", "Extra ➕"],
    "Skills": [", ".join(matched_skills) or "None",
               ", ".join(missing_skills) or "None",
               ", ".join(extra_skills) or "None"]
})
skills_table


**What this cell does:** Calculates what percentage of the job's required skills you already have, and displays everything in a table.
**Why we use it:** A single percentage number is an easy way to summarize the skill comparison at a glance.
**Expected output:** A printed percentage line, followed by a 3-row table.


---
## 🧬 PHASE 7 — Semantic Similarity (AI Embeddings)

### Key concepts (beginner explanation)

- **Embeddings:** A way of turning a sentence into a long list of numbers (a "vector") that captures its *meaning*, not just its exact words. Sentences with similar meaning end up with similar numbers.
- **Vector Representation:** Think of an embedding as GPS coordinates for meaning — two sentences that mean similar things will have coordinates that are close together, even if they use different words (e.g., "built machine learning models" and "developed AI systems" would be *close* in this space).
- **Semantic Search:** Searching or comparing by *meaning* instead of exact keyword matching. This is more powerful than our simple skill-keyword search from Phase 6, because it also picks up on skills/experience described in different words.

We'll use a free, open-source model from the `sentence-transformers` library to do this.


### Cell 24: Load a sentence embedding model

In [ ]:
from sentence_transformers import SentenceTransformer

# 'all-MiniLM-L6-v2' is a small, fast, free open-source embedding model - great for beginners
model = SentenceTransformer('all-MiniLM-L6-v2')

print("✅ Embedding model loaded")


**What this cell does:** Downloads and loads a small pretrained AI model that converts text into embeddings.
**Why we use it:** We don't need to train our own AI model from scratch — we reuse one that's already trained on huge amounts of text (this is called "transfer learning").
**Expected output:** A short download progress bar, then `✅ Embedding model loaded`.


### Cell 25: Generate embeddings for resume and job description

In [ ]:
# We use the ORIGINAL (uncleaned) text here, because embedding models understand full sentences
# and grammar better than a bag of lemmatized keywords.

resume_embedding = model.encode(resume_text)
jd_embedding = model.encode(jd_text)

print("Resume embedding shape:", resume_embedding.shape)
print("Job description embedding shape:", jd_embedding.shape)


**What this cell does:** Converts the full resume and job description text into two number vectors (embeddings).
**Why we use it:** Once text is turned into numbers, we can mathematically measure how "close" two pieces of text are in meaning.
**Expected output:** Something like `Resume embedding shape: (384,)` — a list of 384 numbers representing the meaning of the text.


### Cell 26: Calculate similarity score

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# cosine_similarity compares the "angle" between two vectors: 1.0 = identical meaning, 0 = unrelated
similarity = cosine_similarity([resume_embedding], [jd_embedding])[0][0]
semantic_match_score = round(similarity * 100, 2)

print(f"🧬 Semantic Match Score: {semantic_match_score}%")


**What this cell does:** Uses cosine similarity to compare the resume's and job description's overall meaning, producing one percentage score.
**Why we use it:** This captures overall relevance even when exact keywords differ — e.g. a resume that says "led a team of engineers" is semantically related to a JD asking for "leadership experience," even without an exact keyword match.
**Expected output:** A line like `🧬 Semantic Match Score: 68.42%`


---
## 🏆 PHASE 8 — Candidate Ranking Logic

We now combine everything into **one final score** using a simple weighted formula:

```
Final Score = (Skill Match % x 0.4)
            + (Semantic Similarity % x 0.3)
            + (Experience Score x 0.2)
            + (Education Score x 0.1)
```

**Why these weights?** This is a simple, adjustable starting formula:
- Skills matter most for a technical role (40%)
- Overall semantic fit matters a lot too (30%)
- Relevant experience matters (20%)
- Formal education is a smaller factor (10%)

Feel free to change these weights based on what matters most for your use case!


### Cell 27: Calculate experience and education sub-scores

In [ ]:
# Experience score: cap at 10 years = 100%, scale linearly below that
experience_score = min(resume_experience_years / 10 * 100, 100)

# Education score: simple beginner rule - 100 if any education keyword found, else 50
education_score = 100 if len(resume_education) > 0 else 50

print(f"Experience Score: {experience_score}%")
print(f"Education Score: {education_score}%")


**What this cell does:** Turns years of experience and education keywords into two simple 0-100 scores.
**Why we use it:** To combine different types of information (skills %, similarity %, years, degree) into a single formula, they all need to be on the same 0-100 scale first.
**Expected output:** Two percentage lines.


### Cell 28: Calculate the final candidate score

In [ ]:
final_score = round(
    (skill_match_score * 0.4) +
    (semantic_match_score * 0.3) +
    (experience_score * 0.2) +
    (education_score * 0.1),
    2
)

print(f"🏆 FINAL CANDIDATE SCORE: {final_score} / 100")

score_breakdown = pd.DataFrame({
    "Component": ["Skill Match", "Semantic Similarity", "Experience", "Education", "FINAL SCORE"],
    "Weight": ["40%", "30%", "20%", "10%", "100%"],
    "Score": [skill_match_score, semantic_match_score, experience_score, education_score, final_score]
})
score_breakdown


**What this cell does:** Combines all four scores using our weighted formula into one final number, and displays the breakdown in a table.
**Why we use it:** This gives recruiters (or you!) a single, explainable number to rank candidates by — and the table shows exactly how it was calculated, which is important for trust and transparency.
**Expected output:** A final score line and a 5-row breakdown table.


---
## 📦 PHASE 8B — Upload and Process Your Own Dataset (Batch Mode)

So far, this notebook has analyzed **one resume vs. one job description** at a time.
This section lets you go further: upload **multiple resumes and/or multiple job
descriptions at once** (a small "dataset"), and the notebook will automatically
run the *entire* pipeline above on **every resume-job combination**, then give
you one clean results table you can download as a CSV.

This is useful if you have:
- A folder of candidate resumes you want to rank against one job opening, or
- A few job openings you want to check your resume(s) against, or
- A sample/practice dataset (like the one provided with this project) you want to explore.

> 💡 You can upload `.pdf` or `.txt` files. You can select **multiple files at once**
> in the upload dialog (Ctrl/Cmd-click, or Shift-click for a range).


### Cell: Upload multiple resumes

In [ ]:
from google.colab import files

print("Select ALL the resume files you want to include in your dataset (PDF or TXT).")
print("Tip: you can select multiple files at once in the upload dialog.")
uploaded_resumes_batch = files.upload()

resume_filenames_batch = list(uploaded_resumes_batch.keys())
print(f"\n✅ Uploaded {len(resume_filenames_batch)} resume file(s):")
for f in resume_filenames_batch:
    print(" -", f)


**What this cell does:** Opens a file picker where you can select several resume files at once, and stores all their filenames in a list.
**Why we use it:** Batch analysis needs more than one resume to compare — this is how we collect them.
**Expected output:** A list of every resume filename you uploaded.


### Cell: Upload multiple job descriptions

In [ ]:
print("Select ALL the job description files you want to include in your dataset (PDF or TXT).")
uploaded_jds_batch = files.upload()

jd_filenames_batch = list(uploaded_jds_batch.keys())
print(f"\n✅ Uploaded {len(jd_filenames_batch)} job description file(s):")
for f in jd_filenames_batch:
    print(" -", f)


**What this cell does:** Lets you upload one or more job description files the same way.
**Why we use it:** With multiple resumes AND multiple job descriptions, the notebook can compare every resume against every job (a full "many-to-many" match).
**Expected output:** A list of every job description filename you uploaded.

> 📝 **Tip:** If you only want to test ONE dataset file (e.g. a single CSV you already prepared elsewhere), skip these two upload cells and jump directly to the "Upload a ready-made CSV dataset" cell further below instead.


### 🗜️ (Faster option) Upload your whole dataset as ONE ZIP file

If you already have your datasets organized in folders on your computer —
for example a folder of resumes and a folder of job descriptions — it's
often quicker to zip them up and upload **one file** instead of selecting
many files by hand.

**How to organize your ZIP file (recommended):**
```
my_dataset.zip
├── resumes/
│   ├── resume1.pdf
│   ├── resume2.txt
│   └── ...
└── job_descriptions/
    ├── jd1.pdf
    ├── jd2.txt
    └── ...
```
This is exactly how the sample `TalentForge_AI_Dataset.zip` provided with
this project is structured, so you can test this cell with that file first.

> 📝 If your ZIP doesn't have `resumes/` and `job_descriptions/` folders, the
> cell below will still extract everything and list all the files it found —
> you can then manually set `resume_filenames_batch` and `jd_filenames_batch`
> to whichever filenames belong in each group.


In [ ]:
import zipfile, os

print("Upload a ZIP file containing your dataset (folders of resumes / job descriptions)")
uploaded_zip = files.upload()
zip_filename = list(uploaded_zip.keys())[0]

extract_dir = "my_dataset"
with zipfile.ZipFile(zip_filename, "r") as z:
    z.extractall(extract_dir)

print(f"✅ Extracted '{zip_filename}' into ./{extract_dir}/")

# Walk the extracted folder and collect every PDF/TXT file, grouped by folder name
all_files = []
for root, dirs, filenames in os.walk(extract_dir):
    for fn in filenames:
        if fn.lower().endswith((".pdf", ".txt")):
            all_files.append(os.path.join(root, fn))

print(f"\nFound {len(all_files)} resume/JD files:")
for f in all_files:
    print(" -", f)


**What this cell does:** Uploads a single ZIP file, extracts it into a folder called `my_dataset`, then walks through every subfolder to find all PDF/TXT files inside.
**Why we use it:** Uploading one ZIP is faster and less error-prone than selecting dozens of individual files, especially for larger datasets.
**Expected output:** A confirmation line, then a list of every resume/JD file found inside the ZIP.


In [ ]:
# Automatically split the extracted files into resumes and job descriptions,
# based on whether their path contains "resume" or "job_description" / "jd".
# (This matches the folder-naming convention used in the sample dataset.)

resume_filenames_batch = [f for f in all_files if "resume" in f.lower()]
jd_filenames_batch = [f for f in all_files if "job_description" in f.lower() or "jd" in os.path.basename(f).lower()]

# Fallback: if the automatic split didn't find a clear split (e.g. a flat ZIP with
# no matching folder/file names), put everything in resume_filenames_batch and let
# you edit the two lists manually in this cell before moving on.
if not resume_filenames_batch and not jd_filenames_batch:
    print("⚠️ Could not automatically tell resumes and job descriptions apart.")
    print("   Edit the two lines below to manually assign the files listed above, e.g.:")
    print('   resume_filenames_batch = ["my_dataset/resume1.pdf", "my_dataset/resume2.txt"]')
    print('   jd_filenames_batch = ["my_dataset/jd1.pdf"]')
else:
    print(f"✅ Detected {len(resume_filenames_batch)} resume file(s) and {len(jd_filenames_batch)} job description file(s)")
    print("\nResumes:")
    for f in resume_filenames_batch:
        print(" -", f)
    print("\nJob Descriptions:")
    for f in jd_filenames_batch:
        print(" -", f)


**What this cell does:** Automatically sorts the extracted files into `resume_filenames_batch` and `jd_filenames_batch` — the same two variable names the pipeline cell below expects — based on their folder or file names.
**Why we use it:** This lets the ZIP-upload path plug directly into the same batch pipeline used for the multi-file-picker path, with no extra changes needed.
**Expected output:** A count and list of detected resumes and job descriptions. If detection fails, a note tells you how to fix the two lists by hand.

> ✅ **Either upload path works** — whether you used the multi-file picker above or this ZIP upload, you now have `resume_filenames_batch` and `jd_filenames_batch` ready. Continue to the next cell to run the full pipeline on them.


### Cell: Run the full pipeline on every resume × job-description pair

In [ ]:
import pandas as pd

# This reuses the exact same functions we already defined earlier in the notebook:
# extract_text(), to_lowercase(), remove_punctuation(), remove_special_characters(),
# remove_extra_spaces(), extract_skills(), extract_by_keywords(), model.encode(), etc.

batch_results = []

for r_file in resume_filenames_batch:
    r_raw = extract_text(r_file)
    r_clean = remove_extra_spaces(remove_special_characters(remove_punctuation(to_lowercase(r_raw))))
    r_skills = extract_skills(r_clean)
    r_education = extract_by_keywords(r_clean, EDUCATION_KEYWORDS)
    r_exp_matches = re.findall(r'(\d+)\+?\s*year', r_raw.lower())
    r_exp_years = max([int(x) for x in r_exp_matches], default=0)
    r_embedding = model.encode(r_raw)

    for j_file in jd_filenames_batch:
        j_raw = extract_text(j_file)
        j_clean = remove_extra_spaces(remove_special_characters(remove_punctuation(to_lowercase(j_raw))))
        j_skills = extract_skills(j_clean)
        j_embedding = model.encode(j_raw)

        matched = sorted(set(r_skills) & set(j_skills))
        missing = sorted(set(j_skills) - set(r_skills))
        extra = sorted(set(r_skills) - set(j_skills))

        skill_score = round(len(matched) / len(j_skills) * 100, 2) if j_skills else 0.0
        sim = cosine_similarity([r_embedding], [j_embedding])[0][0]
        semantic_score = round(sim * 100, 2)
        exp_score = min(r_exp_years / 10 * 100, 100)
        edu_score = 100 if len(r_education) > 0 else 50
        final = round(skill_score * 0.4 + semantic_score * 0.3 + exp_score * 0.2 + edu_score * 0.1, 2)

        if final >= 70:
            label = "Good Fit"
        elif final >= 50:
            label = "Moderate Fit"
        else:
            label = "Poor Fit"

        batch_results.append({
            "resume_file": r_file,
            "jd_file": j_file,
            "matched_skills": ", ".join(matched) or "None",
            "missing_skills": ", ".join(missing) or "None",
            "extra_skills": ", ".join(extra) or "None",
            "skill_match_score": skill_score,
            "semantic_similarity_score": semantic_score,
            "experience_years": r_exp_years,
            "final_score": final,
            "match_label": label,
        })

batch_df = pd.DataFrame(batch_results)
print(f"✅ Processed {len(resume_filenames_batch)} resume(s) x {len(jd_filenames_batch)} job description(s) = {len(batch_df)} combinations")
batch_df


**What this cell does:** Loops through every resume and every job description, running the same cleaning → skill extraction → embedding → scoring steps from earlier in the notebook, and collects everything into one results table.
**Why we use it:** This turns your single-pair analysis into a reusable batch pipeline — exactly what you'd need to rank many candidates, or check your resume against many job postings.
**Expected output:** A pandas table with one row per resume-job combination, sorted however it was generated (see the next cell for sorting/ranking).


### Cell: Sort and view the ranked results

In [ ]:
# Sort so the best matches appear first
batch_df_sorted = batch_df.sort_values(by="final_score", ascending=False).reset_index(drop=True)
batch_df_sorted


**What this cell does:** Sorts the batch results table from the highest final score to the lowest.
**Why we use it:** For ranking candidates against a job (or ranking jobs for a candidate), the most useful view is "best match first".
**Expected output:** The same table as before, now ordered by `final_score` descending.


### Cell: Download your results as a CSV file

In [ ]:
output_filename = "talentforge_batch_results.csv"
batch_df_sorted.to_csv(output_filename, index=False)

print(f"✅ Saved results to {output_filename}")
files.download(output_filename)   # triggers a browser download of the CSV


**What this cell does:** Saves the ranked results table as a CSV file and downloads it to your computer.
**Why we use it:** A CSV can be opened in Excel/Google Sheets, shared with a team, or fed into another tool.
**Expected output:** A file named `talentforge_batch_results.csv` downloads through your browser.


### (Alternative) Upload a ready-made CSV dataset instead

If you already have a dataset file (for example, the sample
`resume_job_matching_dataset.csv` provided with this project, or your own
CSV of resume/job data), you can upload and explore it directly instead of
re-running the pipeline.


In [ ]:
print("Upload a CSV dataset file (e.g. resume_job_matching_dataset.csv)")
uploaded_csv = files.upload()
csv_filename = list(uploaded_csv.keys())[0]

my_dataset = pd.read_csv(csv_filename)
print(f"✅ Loaded '{csv_filename}' with {len(my_dataset)} rows and {len(my_dataset.columns)} columns")
my_dataset.head(10)


**What this cell does:** Uploads and loads an existing CSV dataset into a pandas table, and previews the first 10 rows.
**Why we use it:** Sometimes you already have structured data (like the sample dataset that comes with this project) and just want to explore or reuse it, without re-running the whole extraction pipeline.
**Expected output:** A confirmation line with row/column counts, followed by a preview table.


---
## 🤖 PHASE 9 — Generative AI Layer

### Key concept: Prompt Engineering
**Prompt Engineering** means carefully writing the instructions (the "prompt") you give an AI model so it produces the output you want. A good prompt is specific, gives context, and states the exact format you want back.

### Two options for this phase
- **Option A (uses an API key, costs a small amount):** Use the OpenAI API for higher-quality writing.
- **Option B (100% free, no API key):** Use a free Hugging Face open-source text-generation model.

We'll set up **Option A** with a safe fallback: if you don't have an API key, the notebook automatically uses a free template-based generator instead, so the notebook still works either way.


### Cell 29: (Optional) Enter your OpenAI API key

In [ ]:
# If you have an OpenAI API key and want higher-quality AI writing, paste it below between the quotes.
# If you don't have one, just leave it as an empty string "" and skip to the next cell -
# the notebook will use a free fallback generator instead.

OPENAI_API_KEY = ""   # <-- paste your key here, e.g. "sk-..."

use_openai = len(OPENAI_API_KEY.strip()) > 0
print("Using OpenAI API:" , use_openai)


**What this cell does:** Lets you optionally provide an API key for better-quality AI text.
**Why we use it:** This keeps the project flexible — beginners with no budget can still complete the whole project for free.
**Expected output:** `Using OpenAI API: False` (unless you added a real key).


### Cell 30: Build the prompt for the candidate report

In [ ]:
prompt = f"""
You are an expert technical recruiter. Based on the data below, write a concise candidate report.

Matched Skills: {matched_skills}
Missing Skills: {missing_skills}
Extra Skills: {extra_skills}
Semantic Match Score: {semantic_match_score}%
Final Candidate Score: {final_score}/100
Years of Experience: {resume_experience_years}

Write the report with these exact sections:
1. Candidate Summary (2-3 sentences)
2. Resume Strengths (bullet points)
3. Resume Weaknesses (bullet points)
4. Missing Skills (bullet points)
5. Learning Recommendations (bullet points)
6. Best Job Roles (bullet points)
7. Interview Readiness Report (2-3 sentences)
"""

print(prompt)


**What this cell does:** Builds a clear, structured instruction (prompt) that tells the AI exactly what sections to write and gives it the real data to base its answer on.
**Why we use it:** This is prompt engineering in action — specific instructions + real data = a useful, structured AI response instead of a vague one.
**Expected output:** The full prompt text printed out.


### Cell 31: Generate the AI report (with free fallback)

In [ ]:
def free_fallback_report():
    """A simple, free, template-based report generator.
    Used automatically when no OpenAI API key is provided."""
    lines = []
    lines.append("1. CANDIDATE SUMMARY")
    lines.append(f"This candidate has {resume_experience_years} years of experience and matches "
                 f"{skill_match_score}% of the required skills, with an overall semantic fit of "
                 f"{semantic_match_score}%.\n")

    lines.append("2. RESUME STRENGTHS")
    for s in (matched_skills + extra_skills)[:8] or ["No strong keyword matches found"]:
        lines.append(f"- Demonstrated experience with {s}")
    lines.append("")

    lines.append("3. RESUME WEAKNESSES")
    lines.append("- Resume could better highlight quantifiable achievements and outcomes")
    lines.append("")

    lines.append("4. MISSING SKILLS")
    for s in missing_skills or ["None - great skill coverage!"]:
        lines.append(f"- {s}")
    lines.append("")

    lines.append("5. LEARNING RECOMMENDATIONS")
    for s in missing_skills[:5] or ["Keep building projects in your current stack"]:
        lines.append(f"- Take an online course or build a project using {s}")
    lines.append("")

    lines.append("6. BEST JOB ROLES")
    lines.append("- Data Analyst\n- NLP Intern\n- AI Intern\n- Data Evaluation Intern")
    lines.append("")

    lines.append("7. INTERVIEW READINESS REPORT")
    readiness = "high" if final_score >= 70 else "moderate" if final_score >= 50 else "developing"
    lines.append(f"Based on a final score of {final_score}/100, this candidate's interview readiness is {readiness}.")

    return "\n".join(lines)


if use_openai:
    # This block only runs if you provided a real API key in Cell 29
    import requests
    headers = {"Authorization": f"Bearer {OPENAI_API_KEY}", "Content-Type": "application/json"}
    body = {"model": "gpt-4o-mini", "messages": [{"role": "user", "content": prompt}]}
    response = requests.post("https://api.openai.com/v1/chat/completions", headers=headers, json=body)
    ai_report = response.json()["choices"][0]["message"]["content"]
else:
    ai_report = free_fallback_report()

print(ai_report)


**What this cell does:** Generates the 7-section candidate report — using the real OpenAI API if you added a key, or a free rule-based generator if you didn't.
**Why we use it:** This shows you both paths: how real GenAI API calls work, and a free way to get similar structured output while learning.
**Expected output:** A neatly formatted text report with all 7 numbered sections.


---
## ❓ PHASE 10 — Interview Question Generator

We'll generate interview questions tailored to the skills found in the resume — technical (SQL, Python, Power BI) and behavioral.


### Cell 32: Question bank by skill

In [ ]:
# A small beginner-friendly question bank. Add more questions/skills as you like!

QUESTION_BANK = {
    "python": ["What is the difference between a list and a tuple in Python?",
               "How do you handle exceptions in Python?"],
    "sql": ["What is the difference between INNER JOIN and LEFT JOIN?",
            "How would you find duplicate rows in a table?"],
    "power bi": ["What is the difference between a measure and a calculated column?",
                 "How do you optimize a slow Power BI report?"],
    "machine learning": ["What is overfitting and how do you prevent it?",
                          "Explain the difference between supervised and unsupervised learning."],
    "excel": ["What is the difference between VLOOKUP and INDEX-MATCH?",
              "How do you use a pivot table to summarize data?"],
}

BEHAVIORAL_QUESTIONS = [
    "Tell me about a project you're proud of and your role in it.",
    "Describe a time you faced a difficult problem at work. How did you solve it?",
    "How do you prioritize tasks when working on multiple projects?",
]

print("✅ Question bank ready")


**What this cell does:** Defines a dictionary mapping each skill to a couple of relevant interview questions, plus a list of general behavioral questions.
**Why we use it:** Simple lookups like this are easy to understand and easy to expand later.
**Expected output:** `✅ Question bank ready`


### Cell 33: Generate questions based on the candidate's skills

In [ ]:
generated_questions = []

for skill in resume_skills:
    if skill in QUESTION_BANK:
        generated_questions.extend(QUESTION_BANK[skill])

print("----- TECHNICAL QUESTIONS -----")
for q in generated_questions:
    print("-", q)

print("\n----- BEHAVIORAL QUESTIONS -----")
for q in BEHAVIORAL_QUESTIONS:
    print("-", q)


**What this cell does:** Loops through the candidate's detected skills and pulls out matching interview questions, plus always includes general behavioral questions.
**Why we use it:** This personalizes the interview prep to the actual candidate instead of using a generic list.
**Expected output:** Two printed lists of questions, technical and behavioral.


---
## 📋 PHASE 11 — Explainable AI Report

**Explainable AI** means showing *why* the AI produced a certain result, not just the result itself. Recruiters (and candidates!) trust a score much more when they can see the reasoning behind it.


### Cell 34: Build the final explainable report

In [ ]:
final_report = f"""
==================== EXPLAINABLE AI REPORT ====================

FINAL SCORE: {final_score} / 100

WHY THE CANDIDATE MATCHED:
- Matched {len(matched_skills)} of {len(jd_skills_set)} required skills: {matched_skills}
- Semantic (meaning-based) similarity to job description: {semantic_match_score}%
- {resume_experience_years} years of relevant experience detected

WHY THE CANDIDATE DID NOT FULLY MATCH:
- Missing skills: {missing_skills if missing_skills else 'None'}

RECOMMENDED CERTIFICATIONS:
{[f'Certification in {s}' for s in missing_skills[:3]] if missing_skills else ['None needed - strong match']}

SUGGESTED PROJECTS:
{[f'Build a small project using {s}' for s in missing_skills[:3]] if missing_skills else ['Keep building on current strengths']}

CAREER ROADMAP:
1. Close the missing-skill gaps listed above
2. Build 1-2 portfolio projects showing those skills
3. Apply to roles similar to: Data Analyst, NLP Intern, AI Intern
4. Practice the generated interview questions

=================================================================
"""

print(final_report)


**What this cell does:** Assembles everything we've calculated into one final, human-readable, explainable report.
**Why we use it:** This is the actual deliverable a recruiter or candidate would read — it explains match, gaps, and next steps clearly.
**Expected output:** A formatted text block with all sections filled in based on your resume and job description.


---
## ⚖️ PHASE 12 — Responsible AI

### Why Responsible AI matters
When AI helps make decisions about people's careers, it must be used carefully:
- **Bias Detection:** Checking that the tool doesn't unfairly favor or penalize candidates based on things like name, gender, age, or background — instead of actual skills and experience.
- **Fair Hiring:** Making sure the scoring criteria are based on job-relevant factors only.
- **Ethical AI:** Being transparent about how a decision (like a score) was reached, and never letting AI make a final hiring decision alone — a human should always review it.
- **Data Privacy:** Resumes contain personal information (name, address, phone number). This data should be handled carefully, not stored or shared without consent, and deleted when no longer needed.

This project is a **decision-support tool**, not a decision-maker. A human recruiter should always review the final output.


### Cell 35: Basic responsible-AI checks

In [ ]:
responsible_ai_notes = []

# Check 1: does our scoring formula use any protected/personal attributes? (it shouldn't, and doesn't)
responsible_ai_notes.append("✅ Scoring formula uses only skills, experience, education, and semantic "
                             "similarity - no name, age, gender, or personal attributes are used.")

# Check 2: is the score explainable? (yes - we showed the breakdown in Phase 8 and 11)
responsible_ai_notes.append("✅ Score breakdown is fully visible to the user (see Phase 8 & 11).")

# Check 3: reminder that a human should review this
responsible_ai_notes.append("⚠️ This tool should support, not replace, a human recruiter's judgment.")

for note in responsible_ai_notes:
    print(note)


**What this cell does:** Runs a few simple, beginner-level checks and reminders about responsible use of this AI tool.
**Why we use it:** Even a small project should build good habits around fairness and transparency.
**Expected output:** Three printed notes confirming responsible design choices.


---
## 🖥️ PHASE 13 — Simple Dashboard (Streamlit)

Colab is great for step-by-step learning, but a **dashboard** lets anyone use your project through a simple web page instead of running code cells.

We'll write a Streamlit app (`app.py`) with these sections:
- Upload Resume
- Upload Job Description
- Match Score
- Skills Match
- Missing Skills
- AI Recommendations

> 📝 **Note:** Streamlit apps don't run *inside* a notebook cell the normal way — they run as a separate web app. The next cell writes the `app.py` file for you. To actually view it, you'd run it locally (`streamlit run app.py`) or use a tool like `localtunnel`/`ngrok` from Colab to get a public link.


In [ ]:
%%writefile app.py
# This writes everything below into a file called app.py
# Run it later with:  streamlit run app.py

import streamlit as st
import re, string
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

st.set_page_config(page_title="Resume Intelligence Copilot", layout="wide")
st.title("🧠 Resume Intelligence and Job-Matching Copilot")

SKILLS_DB = ["python", "sql", "excel", "power bi", "tableau", "machine learning",
             "deep learning", "nlp", "data analysis", "pandas", "numpy",
             "scikit-learn", "tensorflow", "aws", "azure", "git", "communication"]

def clean_text(text):
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def extract_skills(text):
    text = text.lower()
    return sorted(set([s for s in SKILLS_DB if s in text]))

@st.cache_resource
def load_model():
    return SentenceTransformer('all-MiniLM-L6-v2')

model = load_model()

# ---- Upload Resume ----
st.header("1. Upload Resume")
resume_file = st.file_uploader("Upload your resume (.txt)", type=["txt"])

# ---- Upload Job Description ----
st.header("2. Upload Job Description")
jd_file = st.file_uploader("Upload the job description (.txt)", type=["txt"])

if resume_file and jd_file:
    resume_text = resume_file.read().decode("utf-8", errors="ignore")
    jd_text = jd_file.read().decode("utf-8", errors="ignore")

    resume_clean = clean_text(resume_text)
    jd_clean = clean_text(jd_text)

    resume_skills = extract_skills(resume_clean)
    jd_skills = extract_skills(jd_clean)

    matched = sorted(set(resume_skills) & set(jd_skills))
    missing = sorted(set(jd_skills) - set(resume_skills))

    skill_score = round(len(matched) / len(jd_skills) * 100, 2) if jd_skills else 0
    sim = cosine_similarity([model.encode(resume_text)], [model.encode(jd_text)])[0][0]
    semantic_score = round(sim * 100, 2)
    final_score = round(skill_score * 0.6 + semantic_score * 0.4, 2)

    # ---- Match Score ----
    st.header("3. Match Score")
    st.metric("Final Match Score", f"{final_score}%")

    # ---- Skills Match ----
    st.header("4. Skills Match")
    st.write("✅ Matched Skills:", matched)

    # ---- Missing Skills ----
    st.header("5. Missing Skills")
    st.write("❌ Missing Skills:", missing)

    # ---- AI Recommendations ----
    st.header("6. AI Recommendations")
    for skill in missing:
        st.write(f"- Consider learning **{skill}** to improve your match for this role.")
else:
    st.info("Upload both a resume and a job description (.txt files) to see results.")


**What this cell does:** Writes a complete, working Streamlit web app to a file called `app.py` in your Colab environment (the `%%writefile` command saves everything below it into that file instead of running it).
**Why we use it:** This turns your notebook logic into a shareable, click-and-use web page instead of code cells.
**Expected output:** `Writing app.py`


### Cell: Deployment steps (explanation)

**How to actually run and view this dashboard:**

1. **Locally on your computer (easiest):**
   - Download `app.py` from Colab's file browser (left sidebar → files icon).
   - On your computer, open a terminal in that folder and run:
     ```
     pip install streamlit sentence-transformers scikit-learn
     streamlit run app.py
     ```
   - It opens automatically in your browser at `http://localhost:8501`.

2. **From inside Colab (using a public tunnel):**
   ```python
   !pip install streamlit -q
   !npm install -g localtunnel -q
   !streamlit run app.py &>/content/logs.txt &
   !npx localtunnel --port 8501
   ```
   This prints a public URL you can click to view the dashboard, even though it's running inside Colab.

3. **Free cloud hosting (for a permanent link):** Push your project to GitHub, then deploy for free at [streamlit.io/cloud](https://streamlit.io/cloud) by connecting your GitHub repo.


---
## 🗂️ PHASE 14 — Final Project Structure

Once you're ready to move this from Colab into a real GitHub project, organize your files like this:

```
resume-intelligence-copilot/
│
├── data/
│   ├── resumes/                # sample resume files (PDF/TXT)
│   └── job_descriptions/       # sample job description files
│
├── notebooks/
│   └── resume_intelligence_copilot.ipynb   # this Colab notebook
│
├── models/                     # (optional) any saved/fine-tuned models
│
├── outputs/                    # generated reports, match scores, charts
│
├── app.py                      # the Streamlit dashboard (Phase 13)
├── requirements.txt            # list of Python libraries needed
└── README.md                   # project overview, setup steps, screenshots
```


In [ ]:
# This cell creates the folder structure above right here in Colab,
# so you can see it and start filling it in.

import os

folders = [
    "resume-intelligence-copilot/data/resumes",
    "resume-intelligence-copilot/data/job_descriptions",
    "resume-intelligence-copilot/notebooks",
    "resume-intelligence-copilot/models",
    "resume-intelligence-copilot/outputs",
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)   # exist_ok=True means "don't error if it already exists"

print("✅ Project folder structure created:")
for folder in folders:
    print(" -", folder)


**What this cell does:** Creates the actual folders for the project structure shown above, directly inside your Colab session.
**Why we use it:** Seeing (and building) the real folder structure makes it much easier to later copy this project into a proper GitHub repository.
**Expected output:** A list of the 5 folders that were created.


In [ ]:
%%writefile resume-intelligence-copilot/requirements.txt
pdfplumber
nltk
sentence-transformers
scikit-learn
pandas
streamlit


**What this cell does:** Creates a `requirements.txt` file listing every library this project needs.
**Why we use it:** Anyone who downloads your project can install everything at once with `pip install -r requirements.txt`. This is standard practice for real, shareable projects.
**Expected output:** `Writing resume-intelligence-copilot/requirements.txt`


---
## 🎤 PHASE 15 — Resume & Interview Preparation

Use the text below when you talk or write about this project — for your resume, GitHub, LinkedIn, or interviews.


### 📄 Project Description (for your Resume)

> **Resume Intelligence and Job-Matching Copilot** — Built an end-to-end NLP and Generative AI system in Python that parses resumes and job descriptions, extracts skills/education/experience using keyword and pattern-based information extraction, computes semantic similarity using Sentence-Transformer embeddings, and generates a weighted candidate match score. Integrated a Generative AI layer for automated candidate summaries, skill-gap analysis, and personalized interview question generation, and built a Streamlit dashboard for interactive use.


### 📘 GitHub README (starter template)

```markdown
# Resume Intelligence and Job-Matching Copilot

An end-to-end NLP + Generative AI project that analyzes resumes against job
descriptions, calculates a match score, identifies missing skills, and
generates AI-powered interview questions and candidate reports.

## Features
- Resume & job description text extraction (PDF/TXT)
- NLP preprocessing (tokenization, stopword removal, lemmatization)
- Keyword-based information extraction (skills, education, experience)
- Skill gap analysis (matched / missing / extra skills)
- Semantic similarity scoring using Sentence-Transformer embeddings
- Weighted candidate ranking formula
- Generative AI candidate report (summary, strengths, weaknesses, recommendations)
- Personalized interview question generator
- Explainable AI report
- Streamlit dashboard

## Tech Stack
Python, NLTK, Sentence-Transformers, scikit-learn, pandas, Streamlit

## How to Run
1. Clone this repo
2. `pip install -r requirements.txt`
3. Open `notebooks/resume_intelligence_copilot.ipynb` in Jupyter/Colab, OR
4. Run the dashboard: `streamlit run app.py`

## Project Structure
See `notebooks/resume_intelligence_copilot.ipynb` (Phase 14) for the full folder layout.

## Future Improvements
- Replace keyword-based skill extraction with a trained NER model
- Support DOCX resumes
- Add multi-resume batch ranking
- Add a bias/fairness audit dashboard
- Fine-tune a small open-source LLM for the report-generation step
```


### 💼 LinkedIn Project Description

> 🚀 Excited to share a project I built: **Resume Intelligence and Job-Matching Copilot** — an AI recruitment assistant that reads resumes and job descriptions, scores candidate-job fit using NLP + Sentence-Transformer embeddings, flags missing skills, and uses Generative AI to write candidate summaries and personalized interview questions. Built with Python, NLTK, Sentence-Transformers, and Streamlit. #NLP #GenerativeAI #MachineLearning #DataScience


### 🗣️ HR Interview Explanation (simple, non-technical)

> "I built a tool that acts like an assistant for recruiters. You give it a resume and a job posting, and it tells you how well they match, what skills are missing, and even suggests interview questions — all automatically, using AI. It saves a recruiter time when they're screening a lot of resumes."


### 🧑‍💻 Technical Interview Explanation

> "The pipeline starts with text extraction from PDF/TXT files, followed by standard NLP preprocessing — lowercasing, punctuation removal, tokenization, stopword removal, and lemmatization using NLTK. I extract structured information (skills, education, experience) using keyword and regex-based matching. For deeper comparison, I use a pretrained Sentence-Transformer model (`all-MiniLM-L6-v2`) to generate embeddings for the resume and job description, then compute cosine similarity to get a semantic match score. I combine skill match %, semantic similarity, experience, and education into a weighted final score. For the generative layer, I use prompt engineering to instruct an LLM (with a free rule-based fallback) to produce a structured candidate report and interview questions. The whole thing is wrapped in a Streamlit dashboard for non-technical users."


### 🏗️ Project Architecture Explanation (with diagram)


```mermaid
flowchart TD
    A[Resume File] --> C[Text Extraction]
    B[Job Description File] --> C
    C --> D[Text Cleaning]
    D --> E[NLP Processing\nTokenize / Stopwords / Lemmatize]
    E --> F[Information Extraction\nSkills / Education / Experience]
    F --> G[Skill Matching\nMatched / Missing / Extra]
    E --> H[Semantic Similarity\nEmbeddings + Cosine Similarity]
    G --> I[Candidate Ranking\nWeighted Final Score]
    H --> I
    I --> J[Generative AI Layer\nSummary / Strengths / Weaknesses]
    I --> K[Interview Question Generator]
    J --> L[Explainable AI Report]
    K --> L
    L --> M[Streamlit Dashboard]
```


### 🔮 Future Improvements

- Replace simple keyword matching with a fine-tuned **NER model** for more accurate skill/entity extraction
- Support **DOCX** resumes in addition to PDF/TXT
- Add **batch mode** to rank many resumes against one job description at once
- Add a proper **bias and fairness audit** dashboard (e.g., checking score distributions across anonymized groups)
- Fine-tune or use a larger open-source LLM (e.g., via Hugging Face) for even better report writing
- Add authentication and resume storage with privacy controls for a production version


---
## 🎉 You're done!

You've now built a complete, explainable, beginner-friendly **NLP + Generative AI** project from raw resume text all the way to an AI-generated candidate report, interview questions, and a Streamlit dashboard.

**Next step:** Run every cell from top to bottom in Google Colab, upload your own resume and a real job description, and read each explanation as you go.
